# Rafa request — fixes and prior C/O adjustments
Read-only analysis. Two verified correction examples on a map; unique fix counts for Ithaca, Canandaigua and Fillmore, identifying prior C/O adjustment evidence separately. Set the snapshot paths below to the local files from the same run. No synthetic data or automatic pipeline execution.

In [ ]:
from pathlib import Path
import pandas as pd
import folium
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "src" / "mtx").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "src" / "mtx").exists(), "Run from a SmartLink repository checkout"

DIVISIONS = ["Ithaca", "Canandaigua", "Fillmore"]
REVIEW_ROOT = ROOT / "data/processed/final_review"
audits = sorted(REVIEW_ROOT.glob("*/*/all_member_audit.parquet"))
assert audits, f"No final-review audit under {REVIEW_ROOT}"
AUDIT = audits[-1]  # change explicitly if the latest is not the intended run

# Use SAME-RUN full AMI and transformer snapshots (not the filtered candidate subset).
AMI_FILE = ROOT / "data/cache/ami/ny_ami.parquet"  # adjust to your actual AMI snapshot
TX_FILE = ROOT / "data/cache/transformers/ny_transformers.parquet"  # adjust
# Optional: curated historical C/O adjustment evidence, one row per affected meter.
# Must contain meter_number and a documented C/O adjustment field; do not infer from 'needs CO'.
CO_HISTORY_FILE = None  # e.g. ROOT / "data/reference/co_adjustments.csv"

def read(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Set the correct snapshot path: {path}")
    return pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path, dtype="string")

def rename(frame, aliases):
    frame = frame.copy()
    for target, choices in aliases.items():
        if target not in frame:
            source = next((c for c in choices if c in frame), None)
            if source: frame = frame.rename(columns={source: target})
    return frame

audit = rename(read(AUDIT), {"division": ["Division"], "meter_number": ["MeterID"],
    "target": ["proposed_final_transformer", "Final_Recommended_FLOC", "recommended_transformer"],
    "current": ["current_transformer", "current_meter_floc", "SuperiorFLOC"]})
required = {"meter_number", "division", "status", "target", "current"}
assert required <= set(audit), f"Audit missing: {required - set(audit)}"
for c in ["meter_number", "target", "current"]:
    audit[c] = audit[c].astype("string").str.strip().str.upper()
audit["division"] = audit["division"].astype("string").str.strip().str.title()
audit = audit[audit.division.isin(DIVISIONS)].copy()
audit["is_fix"] = audit.target.notna() & audit.current.notna() & audit.target.ne(audit.current)
# Only accepted corrections, not held/rejected or already-sent records.
accepted = audit.status.isin(["CORRECT", "NEEDS_CONNECTION_OBJECT"])
if "already_sent" in audit: accepted &= ~audit.already_sent.fillna(False).astype(bool)
fixes = audit[accepted & audit.is_fix].copy()
assert not fixes.meter_number.duplicated().any(), "Audit contains duplicate meters: reconcile before counting"

if CO_HISTORY_FILE is not None:
    history = rename(read(CO_HISTORY_FILE), {"meter_number": ["MeterID"]})
    assert "meter_number" in history, "C/O history must identify meter_number"
    history["meter_number"] = history.meter_number.astype("string").str.strip().str.upper()
    co_ids = set(history.meter_number.dropna())
    fixes["prior_co_adjustment"] = fixes.meter_number.isin(co_ids)
else:
    fixes["prior_co_adjustment"] = pd.NA  # unknown, NOT false
print("Final review:", AUDIT)
print("Unique accepted fixes:", len(fixes))
print("Historical C/O evidence:", CO_HISTORY_FILE or "NOT PROVIDED (unknown)")


In [ ]:
# Unique meter totals by division; prior C/O counts only when evidenced.
summary = (fixes.groupby("division").meter_number.nunique()
           .reindex(DIVISIONS, fill_value=0).rename("total_unique_fixes").to_frame())
if CO_HISTORY_FILE is not None:
    summary["fixes_with_prior_co_adjustment"] = (fixes[fixes.prior_co_adjustment]
        .groupby("division").meter_number.nunique().reindex(DIVISIONS, fill_value=0))
else:
    summary["fixes_with_prior_co_adjustment"] = "UNKNOWN — provide CO_HISTORY_FILE"
display(summary)


In [ ]:
# Pick two short-distance accepted changes; show current and proposed transformers.
ami = rename(read(AMI_FILE), {"meter_number": ["MeterID"], "lat": ["LATITUDE", "latitude"],
    "lon": ["LONGITUDE", "longitude"]})
tx = rename(read(TX_FILE), {"floc": ["FuncLoc", "candidate_transformer", "transformer_floc"],
    "lat": ["Location_Y", "latitude"], "lon": ["Location_X", "longitude"]})
assert {"meter_number", "lat", "lon"} <= set(ami)
assert {"floc", "lat", "lon"} <= set(tx)
ami["meter_number"] = ami.meter_number.astype("string").str.strip().str.upper()
tx["floc"] = tx.floc.astype("string").str.strip().str.upper()
for df in (ami, tx):
    for c in ("lat", "lon"): df[c] = pd.to_numeric(df[c], errors="coerce")
assert not ami.meter_number.duplicated().any(), "AMI has duplicate meter IDs"
assert not tx.floc.duplicated().any(), "Transformer FLOC has multiple coordinate records"
examples = fixes.merge(ami[["meter_number", "lat", "lon"]], on="meter_number", validate="one_to_one")
for role, key in [("old", "current"), ("new", "target")]:
    points = tx.rename(columns={"floc": key, "lat": f"{role}_lat", "lon": f"{role}_lon"})
    examples = examples.merge(points, on=key, how="left", validate="many_to_one")
needed = ["lat", "lon", "old_lat", "old_lon", "new_lat", "new_lon"]
examples = examples.dropna(subset=needed)
examples = examples[examples[needed[::2]].abs().le(180).all(axis=1)]
if "proposed_distance_ft" in examples:
    examples = examples[pd.to_numeric(examples.proposed_distance_ft, errors="coerce").le(550)]
examples = examples.sort_values("proposed_distance_ft" if "proposed_distance_ft" in examples else "meter_number").head(2)
assert len(examples) == 2, "Fewer than two accepted fixes with complete coordinates; check snapshots"
m = folium.Map(location=[examples.lat.mean(), examples.lon.mean()], zoom_start=12)
for row in examples.itertuples():
    folium.Marker([row.lat, row.lon], tooltip=f"Meter {row.meter_number}",
                  icon=folium.Icon(color="blue")).add_to(m)
    folium.Marker([row.old_lat, row.old_lon], tooltip=f"Previous TX {row.current}",
                  icon=folium.Icon(color="red")).add_to(m)
    folium.Marker([row.new_lat, row.new_lon], tooltip=f"Corrected TX {row.target}",
                  icon=folium.Icon(color="green")).add_to(m)
    folium.PolyLine([[row.lat,row.lon],[row.old_lat,row.old_lon]], color="red", dash_array="5,5").add_to(m)
    folium.PolyLine([[row.lat,row.lon],[row.new_lat,row.new_lon]], color="green", weight=4).add_to(m)
display(examples[["meter_number","division","current","target","status","prior_co_adjustment"]])
display(m)
# Optional local HTML for Rafa; no data is committed to GitHub.
# m.save("rafa_two_good_fixes.html")
